# Trabajo Practio de Laboratorio Nº 5 : Contadores


##  Datos del grupo

*   **Curso:** R4052
*   **Nº de Grupo:** 4
*   **Integrantes:**

  1.   Ricardo Condori
  2.   Matías Fernando Gonzalez Falbo
  3.   Facundo Rausch Stola
  4.   Valentin Dorrego




## Inicializacion

In [1]:
# Librerias de pyhon a utilizar
import numpy as np
import math
import IPython
from IPython.display import display, Math
import pandas as pd

In [2]:
# Funciones Auxiliares
# EJMPLO: Estos son algunos ejemplos de funciones
from typing import Any

def incert_A(mediciones):
  s = np.std(mediciones, ddof=1, )  # ddof=1 -> grados de libertad v = 1/n-1
  return s/math.sqrt(len(mediciones))
    
def incert_A_por_grupos(mediciones, tam_grupo=5):
    mediciones = np.array(mediciones)
    
    # 1. Ajustar los datos para que sean divisibles por tam_grupo si es necesario
    n_completos = (len(mediciones) // tam_grupo) * tam_grupo
    datos_recortados = mediciones[:n_completos]
    
    # 2. Agrupar en vectores de 5 elementos
    vectores = datos_recortados.reshape(-1, tam_grupo)
    
    # 3. Obtener la media de cada vector de 5
    medias = np.mean(vectores, axis=1)
    
    # 4. Calcular la incertidumbre Tipo A sobre el conjunto de medias
    u_A = incert_A(medias)
    
    return u_A, medias
    
def incert_B(err_rel, nd, ctas, mediciones):
  return (err_rel/100 + nd/ctas) * np.mean(mediciones) / math.sqrt(3)

def incert_Total(uA, uB):
  return math.sqrt(uA**2 + uB**2)

def calcular_incertidumbre_tipo_b(fi, N, estabilidad_ppm=3.0):
    """
    Calcula la Incertidumbre Tipo B u_B(f) de medición de frecuencia.
    
    Parámetros:
    - fi: Frecuencia medida (en Hz).
    - N:  Número de cuentas / muestra.
    - estabilidad_ppm: Estabilidad de la base de tiempo en ppm (por defecto 3 ppm).
    
    Retorna:
    - u_b: Incertidumbre Tipo B u_B(f) en Hz.
    """
    # Estabilidad en valor adimensional (ej. 3 ppm -> 3e-6)
    delta_fBT_fBT = estabilidad_ppm * 1e-6
    
    # Error absoluto delta_fi
    delta_fi = (delta_fBT_fBT + (1.0 / N)) * fi
    
    # Incertidumbre Tipo B: u_B(f) = delta_fi / sqrt(3)
    u_b = delta_fi / math.sqrt(3)
    
    return u_b

## Instrumentacion utilizada

Contador universal  utilizado en la practica: **Picotest u6200a**

Generador de funciones **Twintex tfg-3205e**


## Ecuaciones utilizadas durante la 

- Valor mas probable

$$
\overline X = \frac{1}{n} \sum\limits_{i=1}^{n}X_i
$$

- Incertidumbre tipo A
$$
u_{A}(\overline{F_{X}}) = \frac{s(V_{X})}{\sqrt{n}} = \frac{\sqrt{\frac{1}{n-1}\sum\limits_{i=1}^{n} \left( V_{X_{i}} - \overline{V_{X}} \right)^{2}}}{\sqrt{n}} \\[10pt]
$$

- Incertidumbre tipo B, en el caso de la señal cuadrada es la misma tanto para frecuancia como periodo, debido a que el error de disparo es $e_d=0$
$$u_{B}(\overline{F_{X}}) = \frac{\Delta f_I}{f_I} \cdot \frac{1}{\sqrt{3}}= \pm\left(\left|\frac{\Delta f_{BT}}{f_{BT}}\right| + \left|\frac{\Delta N}{N}\right|\right) \cdot \frac{1}{\sqrt{3}}$$

- Incertidumbre combinada

$$u_{C} = \sqrt{u_{A}^2 + u_{B}^2}$$

- Factor de cobertura utilizada

$$
k_{95\%} = 2
$$

## Procedimiento específico de la primera medición

Se conecta la salida del generador de funciones al canal 1 del frecuencímetro. Se selecciona una onda cuadrada y se ajusta la frecuencia del generador a 10 Hz, 1 kHz y 1 MHz, realizando las mediciones para cada uno de estos valores.
En el frecuencímetro se selecciona el modo de medición de frecuencia y se registran las indicaciones obtenidas para los distintos tiempos de gate time: 0,1 s, 1 s y 10 s.
Posteriormente, se selecciona el modo de medición de periodo promediado y se repite el procedimiento utilizando los valores de 1, 10 y 100 ciclos de promediado, registrando los resultados correspondientes.

<div align="center">
    <img src="FrecuencimetroImagenes/1.png" width="500">
</div>



De la hoja de datos del instrumento obtenemos:

- Tiempo de Compuerta $Tc = 10MHz$

<div align="center">
    <img src="FrecuencimetroImagenes/2.png" width="500">
</div>

Estabilidad de la base de tiempos $ \frac{\Delta f_I}{f_I} = 1ppm + 2ppm$ (Asumiendo que su ultima vez calibrado fue hace un 1 año)


<div align="center">
    <img src="FrecuencimetroImagenes/3.png" width="500">
</div>


### Medicion de frecuencia

In [3]:
# Frecuencia Incertidumbre tipo A


datos = {
    "frecuencia": {
        "10Hz": {
            "0.1s": [9.9996717032, 9.9996705519, 9.9996713615, 9.9996711825, 9.9996700963],
            "1s":   [9.999669562790, 9.999669394170, 9.999669369760, 9.999669325830, 9.999669300400],
            "10s":  [9.999668305050, 9.999668080180, 9.999667874500, 9.999667676460, 9.999667476250],
        },
        "1kHz": {
            "0.1s": [999.96657039, 999.96656302, 999.96643485, 999.96655812, 999.96655654],
            "1s":   [999.966482915, 999.966478235, 999.966474496, 999.966472383, 999.966470301],
            "10s":  [999.966432420, 999.966419195, 999.966405975, 999.966392625, 999.966380154],
        },
        "1MHz": {
            "0.1s": [999966.319880, 999966.314478, 999966.312648, 999966.310292, 999966.307393],
            "1s":   [999966.281566, 999966.274257, 999966.272524, 999966.269966, 999966.268025],
            "10s":  [999966.253847, 999966.242899, 999966.233425, 999966.222589, 999966.212885],
        },
        
    }
}

resultados = {}

for frecuencia, mediciones_frecuencia in datos["frecuencia"].items():
    
    resultados[frecuencia] = {}
    
    
    for gate_time, lista_mediciones in mediciones_frecuencia.items():
        
        # Cálculos
        media = np.mean(lista_mediciones)
        u_A = incert_A(lista_mediciones)
        
        # Guardar resultados
        resultados[frecuencia][gate_time] = {
            "mediciones": lista_mediciones,
            "f_prom": media,
            "u_A": u_A
        }

# ==================================================
# INCERTIDUMBRE TIPO B
# ==================================================

u_B = {
    "10Hz": {
        "0.1s": calcular_incertidumbre_tipo_b(9.9996709790, 99996709790),
        "1s":   calcular_incertidumbre_tipo_b(9.9996693910, 99996693910),
        "10s":  calcular_incertidumbre_tipo_b(9.9996678820, 99996678820)
    },

    "1kHz": {
        "0.1s": calcular_incertidumbre_tipo_b(999.96653658, 99996653658),
        "1s":   calcular_incertidumbre_tipo_b(999.96647566, 99996647566),
        "10s":  calcular_incertidumbre_tipo_b(999.96640607, 99996640607)
    },

    "1MHz": {
        "0.1s": calcular_incertidumbre_tipo_b(999966.312938, 999966312938),
        "1s":   calcular_incertidumbre_tipo_b(999966.273267, 999966273267),
        "10s":  calcular_incertidumbre_tipo_b(999966.233129, 999966233129)
    }
}

# ==================================================
# INCERTIDUMBRE COMBINADA
# ==================================================

for frecuencia in resultados:
    
    for gate_time in resultados[frecuencia]:
        
        u_A = resultados[frecuencia][gate_time]["u_A"]
        u_B_actual = u_B[frecuencia][gate_time]
        
        # Combinación de Tipo A y Tipo B
        U = np.sqrt(u_A**2 + u_B_actual**2)*2
        
        # Guardar resultados
        resultados[frecuencia][gate_time]["u_B"] = u_B_actual
        resultados[frecuencia][gate_time]["U"] = U


# ==================================================
# MOSTRAR RESULTADOS
# ==================================================

for frecuencia, datos_frecuencia in resultados.items():
    
    print(f"\n{'='*60}")
    print(f"FRECUENCIA: {frecuencia}")
    print(f"{'='*60}")
    
    for gate_time, datos_gate in datos_frecuencia.items():
        
        print(f"\nGate time: {gate_time}")
        print(f"  f_prom = {datos_gate['f_prom']:.9f} Hz")
        print(f"  u_A    = {datos_gate['u_A']:.3e} Hz")
        print(f"  u_B    = {datos_gate['u_B']:.3e} Hz")
        print(f"  U    = {datos_gate['U']:.3e} Hz")


FRECUENCIA: 10Hz

Gate time: 0.1s
  f_prom = 9.999670979 Hz
  u_A    = 2.893e-07 Hz
  u_B    = 1.732e-05 Hz
  U    = 3.464e-05 Hz

Gate time: 1s
  f_prom = 9.999669391 Hz
  u_A    = 4.606e-08 Hz
  u_B    = 1.732e-05 Hz
  U    = 3.464e-05 Hz

Gate time: 10s
  f_prom = 9.999667882 Hz
  u_A    = 1.458e-07 Hz
  u_B    = 1.732e-05 Hz
  U    = 3.464e-05 Hz

FRECUENCIA: 1kHz

Gate time: 0.1s
  f_prom = 999.966536584 Hz
  u_A    = 2.555e-05 Hz
  u_B    = 1.732e-03 Hz
  U    = 3.464e-03 Hz

Gate time: 1s
  f_prom = 999.966475666 Hz
  u_A    = 2.237e-06 Hz
  u_B    = 1.732e-03 Hz
  U    = 3.464e-03 Hz

Gate time: 10s
  f_prom = 999.966406074 Hz
  u_A    = 9.271e-06 Hz
  u_B    = 1.732e-03 Hz
  U    = 3.464e-03 Hz

FRECUENCIA: 1MHz

Gate time: 0.1s
  f_prom = 999966.312938200 Hz
  u_A    = 2.102e-03 Hz
  u_B    = 1.732e+00 Hz
  U    = 3.464e+00 Hz

Gate time: 1s
  f_prom = 999966.273267600 Hz
  u_A    = 2.332e-03 Hz
  u_B    = 1.732e+00 Hz
  U    = 3.464e+00 Hz

Gate time: 10s
  f_prom = 999966.

### Medicion de periodo

In [4]:

datos = {
    "periodo": {
        "10Hz": {
            "0.1s": [100.00340004, 100.00340016, 100.00340024, 100.00340057, 100.00340077],
            "1s":   [100.003399736, 100.003400325, 100.003400504, 100.003400627, 100.003400736],
            "10s":  [100.003402554, 100.003404101, 100.003405772, 100.003406591, 100.003407400],
        },
        "1kHz": {
            "0.1s": [1.0000341514, 1.0000341529, 1.0000341628, 1.0000342069, 1.0000341560],
            "1s":   [1.00003423038, 1.00003423493, 1.0000342378, 1.00003424158, 1.00003424271],
            "10s":  [1.00003425796, 1.00003426591, 1.00003427706, 1.00003428597, 1.00003429524],
        },
        "1MHz": {
            "0.1s": [0.001000034320950, 0.001000034329300, 0.001000034332770, 0.001000034334330, 0.001000034337490],
            "1s":   [0.001000034363030, 0.001000034370130, 0.001000034374670, 0.001000034377530, 0.001000034379150],
            "10s":  [0.001000034455080, 0.001000034476170, 0.001000034488550, 0.001000034507040, 0.001000034535690],
        },
    }
}



resultados = {}

for periodo, mediciones_periodo in datos["periodo"].items():
    
    resultados[periodo] = {}

    for gate_time, lista_mediciones in mediciones_periodo.items():
        
        # Cálculos
        media = np.mean(lista_mediciones)
        u_A = incert_A(lista_mediciones)
        
        # Guardar resultados
        resultados[periodo][gate_time] = {
            "mediciones": lista_mediciones,
            "f_prom": media,
            "u_A": u_A
        }
        
# ==================================================
# INCERTIDUMBRE TIPO B
# ==================================================

# Nota: como la señal es cuadrada

u_B = {
    "10Hz": {
        "0.1s": calcular_incertidumbre_tipo_b(100.00340035, 10000340035),
        "1s":   calcular_incertidumbre_tipo_b(100.00340038, 10000340038),
        "10s":  calcular_incertidumbre_tipo_b(100.00340528, 10000340528)
    },

    "1kHz": {
        "0.1s": calcular_incertidumbre_tipo_b(1.00003416600, 100003416600),
        "1s":   calcular_incertidumbre_tipo_b(1.00003423747, 100003423700),
        "10s":  calcular_incertidumbre_tipo_b(1.00003427642, 100003427600)
    },

    "1MHz": {
        "0.1s": calcular_incertidumbre_tipo_b(0.00100003433096, 100003433096),
        "1s":   calcular_incertidumbre_tipo_b(0.00100003437290, 100003437290),
        "10s":  calcular_incertidumbre_tipo_b(0.00100003449250, 100003449250)
    }
}


# ==================================================
# INCERTIDUMBRE COMBINADA
# ==================================================

for periodo in resultados:

    for gate_time in resultados[periodo]:

        u_A = resultados[periodo][gate_time]["u_A"]
        u_B_actual = u_B[periodo][gate_time]

        # Combinación de Tipo A y Tipo B
        U = np.sqrt(u_A**2 + u_B_actual**2) * 2

        # Guardar resultados
        resultados[periodo][gate_time]["u_B"] = u_B_actual
        resultados[periodo][gate_time]["U"] = U


# ==================================================
# MOSTRAR RESULTADOS
# ==================================================

for periodo, datos_periodo in resultados.items():

    print(f"\n{'='*60}")
    print(f"FRECUENCIA: {periodo}")
    print(f"{'='*60}")

    for gate_time, datos_gate in datos_periodo.items():

        print(f"\nGate time: {gate_time}")
        print(f"  f_prom = {datos_gate['f_prom']:.9f} ms")
        print(f"  u_A    = {datos_gate['u_A']:.3e} ms")
        print(f"  u_B    = {datos_gate['u_B']:.3e} ms")
        print(f"  U      = {datos_gate['U']:.3e} ms")



FRECUENCIA: 10Hz

Gate time: 0.1s
  f_prom = 100.003400356 ms
  u_A    = 1.358e-07 ms
  u_B    = 1.732e-04 ms
  U      = 3.464e-04 ms

Gate time: 1s
  f_prom = 100.003400386 ms
  u_A    = 1.762e-07 ms
  u_B    = 1.732e-04 ms
  U      = 3.464e-04 ms

Gate time: 10s
  f_prom = 100.003405284 ms
  u_A    = 8.740e-07 ms
  u_B    = 1.732e-04 ms
  U      = 3.464e-04 ms

FRECUENCIA: 1kHz

Gate time: 0.1s
  f_prom = 1.000034166 ms
  u_A    = 1.041e-08 ms
  u_B    = 1.732e-06 ms
  U      = 3.464e-06 ms

Gate time: 1s
  f_prom = 1.000034237 ms
  u_A    = 2.249e-09 ms
  u_B    = 1.732e-06 ms
  U      = 3.464e-06 ms

Gate time: 10s
  f_prom = 1.000034276 ms
  u_A    = 6.697e-09 ms
  u_B    = 1.732e-06 ms
  U      = 3.464e-06 ms

FRECUENCIA: 1MHz

Gate time: 0.1s
  f_prom = 0.001000034 ms
  u_A    = 2.830e-12 ms
  u_B    = 1.732e-09 ms
  U      = 3.464e-09 ms

Gate time: 1s
  f_prom = 0.001000034 ms
  u_A    = 2.904e-12 ms
  u_B    = 1.732e-09 ms
  U      = 3.464e-09 ms

Gate time: 10s
  f_prom = 0

### Extra **Señal senoidal**

Ahora observamos como cambia la estabilidad de la indicación al pasar de señal cuadrada a senoidal.

In [5]:
#Senoidal 1Khz
#Gate time 1s
#Medicion de frecuencia

datos_senoidal = {
    "senoidal": {
        "1kHz": {
            "1s": [
                1.03078381286, 1.02265552819, 1.03317394017, 1.02110388308,
                1.02410388308, 1.01666175811, 1.02027073816, 1.02231930507,
                1.02933610984, 1.04182598697,
            ],
        },
    }
}

resultados_senoidal = {}

for tipo_senal, mediciones_tipo in datos_senoidal["senoidal"].items():

    resultados_senoidal[tipo_senal] = {}

    for gate_time, lista_mediciones in mediciones_tipo.items():

        media = np.mean(lista_mediciones)
        u_A = incert_A(lista_mediciones)

        resultados_senoidal[tipo_senal][gate_time] = {
            "mediciones": lista_mediciones,
            "f_prom": media,
            "u_A": u_A,
        }

# ==================================================
# MOSTRAR RESULTADOS (solo Tipo A, para comparar estabilidad)
# ==================================================

for tipo_senal, datos_tipo in resultados_senoidal.items():
    print(f"\n{'='*60}")
    print(f"SEÑAL: {tipo_senal}")
    print(f"{'='*60}")

    for gate_time, datos_gate in datos_tipo.items():
        print(f"\nGate time: {gate_time}")
        print(f"  f_prom = {datos_gate['f_prom']:.9f} s")
        print(f"  u_A    = {datos_gate['u_A']:.3e} ms")



SEÑAL: 1kHz

Gate time: 1s
  f_prom = 1.026223495 s
  u_A    = 2.375e-03 ms


In [6]:
#Medicion de periodo
datos_senoidal = {
    "senoidal": {
        "1kHz": {
            "1s": [976.068725919, 977.093503919, 984.604540254, 971.948439217, 969.161985529,973.352138287,962.167144999,978.032560121,966.424688244,
                  963.905404533],
        },
    }
}


resultados_senoidal = {}

for tipo_senal, mediciones_tipo in datos_senoidal["senoidal"].items():

    resultados_senoidal[tipo_senal] = {}

    for gate_time, lista_mediciones in mediciones_tipo.items():

        media = np.mean(lista_mediciones)
        u_A = incert_A(lista_mediciones)

        resultados_senoidal[tipo_senal][gate_time] = {
            "mediciones": lista_mediciones,
            "f_prom": media,
            "u_A": u_A,
        }

# ==================================================
# MOSTRAR RESULTADOS (solo Tipo A, para comparar estabilidad)
# ==================================================

for tipo_senal, datos_tipo in resultados_senoidal.items():
    print(f"\n{'='*60}")
    print(f"SEÑAL: {tipo_senal}")
    print(f"{'='*60}")

    for gate_time, datos_gate in datos_tipo.items():
        print(f"\nGate time: {gate_time}")
        print(f"  f_prom = {datos_gate['f_prom']:.9f} us")
        print(f"  u_A    = {datos_gate['u_A']:.3e} us")



SEÑAL: 1kHz

Gate time: 1s
  f_prom = 972.275913102 us
  u_A    = 2.212e+00 us


Existe una diferencia notable, la desviación estándar de la senoidal es mucho mayor que la de la cuadrada.
Esto es debido al punto de disparo en una señal Senoidal es en el cruce por cero. En la señal cuadrada si el punto de disparo es en el flanco de subida o de bajada no cambia el error de disparo, si el RiseTime o FallTime es menor al tiempo de compuerta, el error de disparo será despreciable.

*Señal de entrada senoidal, disparada en la mejor condición (cruce por cero):*

$$V(t) = V_{SP} \sin(\omega t) \quad \Delta V = V_{NP}$$

$$\Delta T = \frac{\Delta V}{\left.\frac{dV}{dt}\right|_{t=t_{TRG}}} = \frac{V_{NP}}{\left.\frac{d[V_{SP}\sin(\omega t)]}{dt}\right|_{t=0}} = \frac{V_{NP}}{\left.V_{SP} \, \omega \cos(\omega t)\right|_{t=0}}$$

$$\Delta T = \frac{V_{NP}}{V_{SP} 2\pi f} \quad \Rightarrow \quad \frac{\Delta T}{T} = \frac{1}{2\pi} \frac{V_{NP}}{V_{SP}}$$

$$e_D = 2 \frac{\Delta T}{T} = \frac{1}{\pi} \frac{V_{NP}}{V_{SP}} \qquad e_D = \frac{1}{\pi \, S/N}$$

Aunque la mayor pendiente de la señal sea en el cruce por cero, su pendiente sigue siendo finita, por eso su error de disparo es mayor y es afectado considerablemente por el ruido.

## Procedimiento específico de la segunda medición
Medimos el ciclo de actividad y el periodo de una serie de señales rectangulares de 1kHz con un 20%, 50% y 80% de dicho ciclo utilizando el modo de medición de intervalos (TI A→B).
Para medir el tiempo en HIGH hay que poner el contador en deteccion por flanco ascendente y descendete y en el caso en que querramos medir el tiempo en LOW hay que poner el contador en dereccion por flanco descendente y ascendente.
<div align="center">
    <img src="FrecuencimetroImagenes/4.png" width="500">
</div>

Para la conexion debemos utilizar un adaptador BNC en T, de tal forma de poder conectar nuestra señal de salida del generador a los dos canales del frecuencimetro y poder utilizar el modo de medicion de intervalos.

In [7]:


# ============================================
# MEDICIONES [µs]
# ============================================

mediciones = {
    "20%": {
        "HIGH": [
            199.2304,
            199.2308,
            199.2309,
            199.2307,
            199.2307
        ],
        "LOW": [
            800.8033,
            800.8035,
            800.8136,
            800.8134,
            800.8034
        ]
    },

    "50%": {
        "HIGH": [
            500.0123,
            500.0223,
            500.0121,
            500.0220,
            500.0122
        ],
        "LOW": [
            500.0123,
            500.0223,
            500.0121,
            500.0220,
            500.0122
        ]
    },

    "80%": {
        "HIGH": [
            799.8333,
            799.8334,
            799.8330,
            799.8335,
            799.8334
        ],
        "LOW": [
            200.2010,
            200.2010,
            200.2011,
            200.2011,
            200.2019
        ]
    }
}


# ============================================
# INCERTIDUMBRE TIPO B [µs]
# ============================================

uB = {
    "20%": {
        "HIGH": calcular_incertidumbre_tipo_b(199.2307, 1992307),
        "LOW":   calcular_incertidumbre_tipo_b(800.8074, 8008074)
    },

    "50%": {
        "HIGH": calcular_incertidumbre_tipo_b(500.0161, 5000161),
        "LOW":   calcular_incertidumbre_tipo_b(500.0161, 5000161)
    },

    "80%": {
        "HIGH": calcular_incertidumbre_tipo_b(799.8333, 7998333),
        "LOW":   calcular_incertidumbre_tipo_b(200.2012, 2002012)
    }
}


# ============================================
# RESULTADOS
# ============================================

for duty, datos in mediciones.items():

    print(f"\n========== DUTY CYCLE {duty} ==========")

    uc_high = None
    uc_low = None

    for tipo, valores in datos.items():

        # Media
        media = np.mean(valores)

        # Desviación estándar
        s = np.std(valores, ddof=1)

        # Incertidumbre Tipo A
        uA = incert_A(valores)

        # Incertidumbre Tipo B
        uB_actual = uB[duty][tipo]

        # Incertidumbre combinada
        uc = math.sqrt(uA**2 + uB_actual**2)

        # Guardamos para calcular el período
        if tipo == "HIGH":
            uc_high = uc
        else:
            uc_low = uc

        print(f"\n{tipo}")
        print(f"  Media                  = {media:.6f} µs")
        print(f"  Desviación estándar    = {s:.6f} µs")
        print(f"  Incertidumbre Tipo A   = {uA:.6f} µs")
        print(f"  Incertidumbre Tipo B   = {uB_actual:.6f} µs")
        print(f"  Incertidumbre combinada = {uc:.6f} µs")


    # ========================================
    # PERÍODO
    # ========================================

    media_high = np.mean(datos["HIGH"])
    media_low = np.mean(datos["LOW"])

    periodo = media_high + media_low

    # Incertidumbre combinada del período
    U_periodo = math.sqrt(uc_high**2 + uc_low**2)*2

    print("\n--------------------------------------------")
    print(f"PERÍODO")
    print(f"  T = {periodo:.6f} µs")
    print(f"  U(T) = {U_periodo:.6f} µs")



========== DUTY CYCLE 20% ==========

HIGH
  Media                  = 199.230700 µs
  Desviación estándar    = 0.000187 µs
  Incertidumbre Tipo A   = 0.000084 µs
  Incertidumbre Tipo B   = 0.000403 µs
  Incertidumbre combinada = 0.000411 µs

LOW
  Media                  = 800.807440 µs
  Desviación estándar    = 0.005533 µs
  Incertidumbre Tipo A   = 0.002474 µs
  Incertidumbre Tipo B   = 0.001445 µs
  Incertidumbre combinada = 0.002865 µs

--------------------------------------------
PERÍODO
  T = 1000.038140 µs
  U(T) = 0.005789 µs

========== DUTY CYCLE 50% ==========

HIGH
  Media                  = 500.016180 µs
  Desviación estándar    = 0.005451 µs
  Incertidumbre Tipo A   = 0.002438 µs
  Incertidumbre Tipo B   = 0.000924 µs
  Incertidumbre combinada = 0.002607 µs

LOW
  Media                  = 500.016180 µs
  Desviación estándar    = 0.005451 µs
  Incertidumbre Tipo A   = 0.002438 µs
  Incertidumbre Tipo B   = 0.000924 µs
  Incertidumbre combinada = 0.002607 µs

-------------

## Procedimiento específico de la tercera medición

Para esta tercera medidicion, mediremos la relación de frecuencia entre una señal senoidal de 10MHz con una de 1MHz utilizando el modo de medición de relación de frecuencias (Ratio A/B) con la mayor resolución posible.

El procedimiento sera muy similar al anterior, debemos conectar la señal de 1Mhz en por el canal B y la señal de 10Mhz en el canal A, esta ultima señal  es obtenida del puerto External 10 MHz Output del contador universal Picotest U6200A.


In [8]:
#Ratio A/B
mediciones = np.array([
    10.0003432990,
    10.0003433141,
    10.0003433312,
    10.0003433324,
    10.0003433258
])

uA = incert_A(mediciones)

print("Media canal B =", np.mean(mediciones))

uB = calcular_incertidumbre_tipo_b(10.0003433204,100003433204)

U = np.sqrt( uA**2 + uB**2)*2

print("U =", U)

Media canal B = 10.000343320499999
U = 3.4642323191125736e-05


In [9]:
#canal B

mediciones = np.array([
999.965658202,
999.965656405,
999.965656546,
999.965656197,
999.965654442
])

uA = incert_A(mediciones)

print("Media canal B =", np.mean(mediciones), "KHz")

uB = calcular_incertidumbre_tipo_b(999.965656358,999965656358)

U = np.sqrt( uA**2 + uB**2)

print("u(fB) = ", U,"KHz")

Media canal B = 999.9656563584 KHz
u(fB) =  0.0017319920029726951 KHz


In [10]:
#canal A
medicion = 10.0000000000

uA = 0

print("Media =", np.mean(medicion), "MHz")

uB = calcular_incertidumbre_tipo_b(10.0000000000,100000000000)

U = np.sqrt( uA**2 + uB**2)

print("u(fA) = ", U,"MHz")

Media = 10.0 MHz
u(fA) =  1.732056581071569e-05 MHz



# Resultados de las mediciónes
> ## Resultados primera medicion
- ### Medicion de frecuencia

### 10Hz

$$
Resultado: f(10Hz)_{Gt = 0.1s} = \left( 9.9996710 \ \pm 3.46 \times 10^{-5} \right)  Hz \qquad k = 2
$$
$$
Resultado: f(10Hz)_{Gt = 1s} = \left( 9.9996694 \ \pm 3.46 \times 10^{-5}  \right)  Hz \qquad k = 2
$$
$$
Resultado: f(10Hz)_{Gt = 10s} = \left( 9.9996679 \ \pm 3.46 \times 10^{-5}  \right) Hz \qquad k = 2
$$

### 1kHz

$$
Resultado: f(1kHz)_{Gt = 0.1s} = \left( 999.96653 \ \pm 3.46 \times 10^{-3}   \right) Hz \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 1s} = \left( 999.96647 \ \pm 3.46 \times 10^{-3} \right)  Hz \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 10s} = \left( 999.96640 \ \pm3.46 \times 10^{-3} \right) Hz \qquad k = 2
$$

### 1MHz

$$
Resultado: f(1kHz)_{Gt = 0.1s} = \left( 999966.31 \ \pm 3.46 \right) Hz \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 1s} = \left( 999966.27 \ \pm 3.464 \right) Hz \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 10s} = \left( 999966.23 \ \pm 3.464 \right) Hz \qquad k = 2
$$

- ### Medicion de periodo

### 10Hz

$$
Resultado: f(10Hz)_{Gt = 0.1s} = \left( 100.003400 \ \pm 3.46 \times 10^{-4} \right)  ms \qquad k = 2
$$
$$
Resultado: f(10Hz)_{Gt = 1s} = \left( 100.003400 \ \pm 3.46 \times 10^{-4} \right)  ms \qquad k = 2
$$
$$
Resultado: f(10Hz)_{Gt = 10s} = \left( 100.003405 \ \pm 3.46 \times 10^{-4} \right) ms \qquad k = 2
$$

### 1kHz

$$
Resultado: f(1kHz)_{Gt = 0.1s} = \left( 1.000034166 \ \pm 3.464 \times 10^{-6}  \right) ms \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 1s} = \left( 1.000034237 \ \pm 3.464 \times 10^{-6} \right)  ms \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 10s} = \left( 1.000034276 \ \pm 3.464 \times 10^{-6} \right) ms \qquad k = 2
$$

### 1MHz

$$
Resultado: f(1kHz)_{Gt = 0.1s} = \left( 1.00003433097 \ \pm 0.0035 \times 10^{-9} \right) us \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 1s} = \left( 1.000034372020 \ \pm 0.0035 \times 10^{-9} \right) us \qquad k = 2
$$
$$
Resultado: f(1kHz)_{Gt = 10s} = \left( 1.000034492506 \ \pm 0.0035 \times 10^{-9} \right) us \qquad k = 2
$$

> ## Resultados segunda medicion

### Duty cycle 20%

$$
T_{D=10\%} = \left( 1000.038140 \ \pm 0.005789 \right)  \mu s \qquad k = 2
$$

### Duty cycle 50%

$$
T_{D=50\%} = \left( 1000.032360 \ \pm 0.007374 \right)  \mu s \qquad k = 2
$$

### Duty cycle 80%

$$
T_{D=80\%} = \left( 1000.034540 \ \pm 0.003022  \times 10^{-5} \right)  \mu s\qquad k = 2
$$

### **Extra:** Efecto de los flancos de detección

Se analizaron las cuatro configuraciones posibles de los flancos de
detección de los canales A y B:

| Configuración | Medición |
|:---:|:---:|
| `++` | $T$ |
| `--` | $T$ |
| `+-` | $T_{HIGH}$ |
| `-+` | $T_{LOW}$ |

Por lo tanto:

$$
T_{++}\approx T_{--}
$$

y

$$
T_{+-}=T_{HIGH}
$$

$$
T_{-+}=T_{LOW}
$$

Además:

$$
T=T_{HIGH}+T_{LOW}
$$

por lo que:

$$
\boxed{T_{+-}+T_{-+}\approx T_{++}\approx T_{--}}
$$

El ciclo de actividad se calcula como:

$$
D=\frac{T_{HIGH}}{T}\times100\%
$$

Para los distintos ciclos de actividad se espera:

$$
D=20\% \Rightarrow T_{HIGH}\approx200\,\mu s,\quad
T_{LOW}\approx800\,\mu s
$$

$$
D=50\% \Rightarrow T_{HIGH}\approx500\,\mu s,\quad
T_{LOW}\approx500\,\mu s
$$

$$
D=80\% \Rightarrow T_{HIGH}\approx800\,\mu s,\quad
T_{LOW}\approx200\,\mu s
$$

> ## Resultados tercera medicion

### Utilizando el modo de medición de relación de frecuencias Ratio A/B

$$
Resultado: Ratio = \left( 10.0003433 \ \pm 3.46 \times 10^{-5} \right) \qquad k = 2
$$

### Relación mediante la división matemática
Se obtiene la incertidumbre combinada

$$
u_C(\bar{Ratio}) = \sqrt{\left[\dfrac{dRatio}{dfA}\right]^2 \cdot u^2(\bar{fA}) + \left[\dfrac{dRatio}{dfB}\right]^2 \cdot u^2(\bar{fB})}
$$

$$
u_C(\bar{Ratio}) = \sqrt{\left[\dfrac{1}{\bar{fB}}\right]^2 \cdot u^2(\bar{fA}) + \left[-\dfrac{\bar{fA}}{\bar{fB}^2}\right]^2 \cdot u^2(\bar{fB})}
$$

$$
u_C(\bar{Ratio}) = \sqrt{\left[\dfrac{1}{999.965656\,KHz}\right]^2 \cdot (1.7320565 \times 10^{-5}\,MHz)^2 + \left[-\dfrac{10MHz}{(999.965656\,KHz)^2}\right]^2 \cdot (0.00173199 KHz)^2}
$$

$$
u_C(\bar{Ratio}) = 3 \times 10^{-4}
$$

$$
U(\bar{Ratio}) = (3 \times 10^{-4}) \cdot K_{95\%} = 6 \times 10^{-4}
$$

$$
Ratio = \left( 10.00034 \pm 0.00060 \right) \qquad k_{95\%} = 2
$$

Para las condiciones de medición utilizadas, el modo Ratio A/B presentó una incertidumbre significativamente menor que la obtenida al calcular la relación a partir de dos mediciones independientes de frecuencia.

# Conclusiones

A partir de las mediciones realizadas se pudo observar como cambia el comportamiento del contador dependiendo del metodo de medicion utilizado.

En las mediciones de frecuencia y periodo se vio que al aumentar el tiempo de gate o la cantidad de ciclos promediados, la medicion se vuelve mas estable y se puede obtener una mejor resolucion, aunque el tiempo necesario para realizar cada medicion tambien aumenta.

Para las mediciones de ciclo de actividad y periodo se pudo observar que la configuracion de los flancos de deteccion influye en el resultado, ya que determina los puntos de la señal entre los cuales se realiza la medicion.

Por ultimo, al comparar la medicion de relacion de frecuencias mediante el modo Ratio A/B con la division de las frecuencias medidas por separado, se obtuvo una menor incertidumbre utilizando directamente el modo Ratio A/B. Esto muestra que para comparar dos frecuencias es conveniente utilizar el modo especifico del contador.

En general el TP permitio ver de forma practica como los distintos parametros y metodos de medicion afectan la estabilidad, resolucion e incertidumbre de las mediciones realizadas con el contador.